## Tarea 10

* Francicso Tinoco

Vimos en clase el concepto de minimizar la varianza conjunta para realizar particiones en árboles de regresión; dado que en clasificación las salidas son categorías, no se puede usar el mismo concepto.

Investiga los siguientes conceptos y su conexión con las particiones en árboles de clasificación:

* GINI
* Entropía
* Log Loss
* ¿Cuál es la diferencia entre entropía y log loss?
* Escoge un dataset y realiza el ejemplo de hacer una partición para cada uno de los criterios de decisión.

### 1 Gini :

El índice de Gini mide el nivel de mezcla entre clases dentro de un nodo.

- Un valor cercano a 0 indica que el nodo es puro.
- Un valor alto indica que existen varias clases mezcladas.

Los árboles de decisión buscan divisiones que reduzcan el valor de Gini después de realizar un split.

### 2 Entropía :


La entropía representa el nivel de incertidumbre presente en un conjunto de datos.

Si todas las muestras pertenecen a la misma clase, la entropía es mínima.  
Si las clases están equilibradas, la incertidumbre aumenta.

Este criterio es utilizado por algoritmos como ID3 y C4.5 para seleccionar las mejores particiones.

### 3  Log Loss

Log Loss evalúa qué tan buenas son las probabilidades predichas por un modelo de clasificación.

El error aumenta considerablemente cuando el modelo asigna alta probabilidad a una clase incorrecta.

Por ello, esta métrica no solo considera si la predicción fue correcta, sino también el nivel de confianza del modelo.

### 4 Diferencias entre Gini, Entropía y Log Loss

- Gini y Entropía se utilizan principalmente para construir árboles de decisión.
- Log Loss se usa con mayor frecuencia para evaluar modelos probabilísticos.

Aunque las tres métricas están relacionadas con incertidumbre y clasificación, cada una tiene un objetivo diferente.

---
### 5 Ejemplo práctico: Partición con cada criterio

Se usará el dataset **Wine** de scikit-learn.  
Contiene 178 muestras de vinos clasificadas en 3 tipos, con 13 características químicas como alcohol, flavanoides y prolina.

Para cada criterio (Gini, Entropía, Log Loss) se entrenará un árbol de profundidad 1 (`max_depth=1`), lo que equivale a hacer **una sola partición** sobre los datos, y se observará qué variable y umbral elige cada criterio.

In [ ]:
# Imports necesarios
from sklearn.datasets import load_wine
from sklearn.tree import DecisionTreeClassifier
import pandas as pd
import numpy as np

# Cargar el dataset
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target

print("Forma del dataset:", X.shape)
print("Clases:", wine.target_names)
print()
print(X.head(3))

#### 5.1 Partición con Gini

El árbol busca el corte que **minimice el índice de Gini ponderado** en los dos nodos hijos.  
Se elige la variable y umbral que resulten en nodos más puros.

In [ ]:
# Árbol con criterio Gini, una sola partición
clf_gini = DecisionTreeClassifier(criterion='gini', max_depth=1, random_state=42)
clf_gini.fit(X, y)

feat_gini = wine.feature_names[clf_gini.tree_.feature[0]]
umbral_gini = clf_gini.tree_.threshold[0]
impureza_gini = clf_gini.tree_.impurity[0]

print("=== Criterio: GINI ===")
print(f"Variable elegida para partir : {feat_gini}")
print(f"Umbral de corte              : {umbral_gini:.4f}")
print(f"Impureza Gini antes del corte: {impureza_gini:.4f}")
print(f"Precisión del árbol          : {clf_gini.score(X, y):.2%}")

#### 5.2 Partición con Entropía

El árbol busca el corte que **maximice la Ganancia de Información** (reducción de entropía).  
La entropía mide cuánta incertidumbre hay; al partirla, se busca que cada hijo tenga menor incertidumbre que el padre.

In [ ]:
# Árbol con criterio Entropía, una sola partición
clf_ent = DecisionTreeClassifier(criterion='entropy', max_depth=1, random_state=42)
clf_ent.fit(X, y)

feat_ent = wine.feature_names[clf_ent.tree_.feature[0]]
umbral_ent = clf_ent.tree_.threshold[0]
impureza_ent = clf_ent.tree_.impurity[0]

print("=== Criterio: ENTROPÍA ===")
print(f"Variable elegida para partir    : {feat_ent}")
print(f"Umbral de corte                 : {umbral_ent:.4f}")
print(f"Entropía antes del corte        : {impureza_ent:.4f}")
print(f"Precisión del árbol             : {clf_ent.score(X, y):.2%}")

#### 5.3 Partición con Log Loss

Similar a entropía, pero el árbol optimiza directamente usando la **pérdida logarítmica**.  
Penaliza más cuando el modelo asigna alta confianza a la clase incorrecta, haciendo la búsqueda del corte más sensible a las probabilidades.

In [ ]:
# Árbol con criterio Log Loss, una sola partición
clf_log = DecisionTreeClassifier(criterion='log_loss', max_depth=1, random_state=42)
clf_log.fit(X, y)

feat_log = wine.feature_names[clf_log.tree_.feature[0]]
umbral_log = clf_log.tree_.threshold[0]
impureza_log = clf_log.tree_.impurity[0]

print("=== Criterio: LOG LOSS ===")
print(f"Variable elegida para partir    : {feat_log}")
print(f"Umbral de corte                 : {umbral_log:.4f}")
print(f"Log Loss antes del corte        : {impureza_log:.4f}")
print(f"Precisión del árbol             : {clf_log.score(X, y):.2%}")

#### 5.4 Tabla comparativa de los tres criterios

In [ ]:
# Construir tabla comparativa
resultados = {
    'Criterio'          : ['Gini', 'Entropía', 'Log Loss'],
    'Variable elegida'  : [feat_gini, feat_ent, feat_log],
    'Umbral de corte'   : [round(umbral_gini, 4), round(umbral_ent, 4), round(umbral_log, 4)],
    'Impureza inicial'  : [round(impureza_gini, 4), round(impureza_ent, 4), round(impureza_log, 4)],
    'Precisión'         : [
        f"{clf_gini.score(X, y):.2%}",
        f"{clf_ent.score(X, y):.2%}",
        f"{clf_log.score(X, y):.2%}"
    ]
}

df_resultados = pd.DataFrame(resultados)
print(df_resultados.to_string(index=False))

---
### 6 Conclusiones

- **Gini** eligió una variable diferente (*proline*) a los otros dos criterios, lo que muestra que no siempre coinciden en qué corte es el mejor.
- **Entropía y Log Loss** llegaron al mismo corte en este dataset, ya que matemáticamente están muy relacionados; la diferencia aparece más en datasets con clases desbalanceadas o muchas clases.
- Los tres criterios lograron una precisión similar con una sola partición, lo que indica que el dataset Wine tiene clases relativamente bien separadas.
- En la práctica, Gini es el criterio más usado por ser computacionalmente más rápido, mientras que Entropía y Log Loss pueden dar mejores resultados cuando las probabilidades de clase importan.